# Trabajo Práctico N.º 2
## Modelado de datos para un medio internacional de noticias: base operativa (OLTP) y almacén dimensional (OLAP) sobre GDELT 2.0 / CAMEO

**Materia:** Introducción a la Ingeniería de Datos — ITBA
**Autor:** Matías Pretel
**Fecha:** 24 de septiembre de 2026

> **Diagramas:** el diagrama Entidad-Relación y el esquema estrella se entregan aparte, en el archivo `TP2.drawio` (página 1: modelo OLTP; página 2: modelo dimensional). Este informe no reproduce los diagramas: los referencia y justifica las decisiones de diseño.

### Declaración de uso de inteligencia artificial

**Tabla 1.** Herramientas de IA utilizadas.

| Herramienta | Modelo | Propósito |
|---|---|---|
| Claude Code | Claude Opus 5.5 | Propuesta inicial de los modelos, generación del XML de draw.io, redacción del informe y código de ejemplo en DuckDB |
| _(completar)_ | _(completar)_ | _(completar)_ |

## 1. Introducción

Un medio internacional de noticias necesita modernizar su plataforma de datos con dos objetivos distintos:

- **Operación periodística (OLTP):** registrar en tiempo real las noticias publicadas, las fuentes consultadas, los actores involucrados y el seguimiento de los eventos en desarrollo.
- **Inteligencia de contenidos (OLAP):** analizar impacto, cobertura y tendencias para tomar decisiones editoriales y de distribución.

Los datos de origen son los eventos de **GDELT 2.0**, codificados con el estándar **CAMEO** (actores, tipo de acción, tono, geolocalización y volumen de cobertura).

**Por qué dos modelos.** Las dos cargas de trabajo tienen necesidades opuestas. El OLTP recibe muchas escrituras pequeñas y concurrentes. Por eso se **normaliza**: cada dato vive en un solo lugar y las restricciones (PK, FK, UNIQUE) protegen la integridad. El OLAP recibe pocas consultas, pero cada una recorre millones de filas y agrupa por muchas dimensiones. Por eso se **desnormaliza** en un esquema estrella, que reduce la cantidad de joins y hace que las consultas sean simples de escribir. Un solo modelo no puede optimizar ambas cosas a la vez.

**Preguntas de negocio.** El almacén debe permitir responder:

1. **Rendimiento e impacto:** ¿qué noticias y temas generan más menciones y artículos (`NumMentions` / `NumArticles`) por país y mes?
2. **Sentimiento de la cobertura:** ¿cómo varía el tono promedio (`AvgTone`) según el tipo de actor involucrado (gobierno, ONG, empresa, etc.)?
3. **Frecuencia y temporalidad:** ¿cómo cambia la intensidad del flujo de noticias entre días hábiles, fines de semana y feriados?
4. **Cobertura multi-temática:** ¿qué eventos requirieron clasificarse bajo varios códigos CAMEO en una misma cobertura?

La consigna anuncia cinco preguntas pero enumera cuatro; el diseño cubre esas cuatro.

## 2. Actividad 1: Modelo operativo (OLTP)

GDELT publica cada evento como una fila de 61 columnas muy desnormalizada: los atributos de Actor1 y Actor2 repetidos, tres bloques de geografía, el código CAMEO en tres niveles y la URL de origen. El modelo operativo separa esa fila en entidades propias.

### 2.1 Entidades

Referencia: `TP2.drawio`, página 1. El modelo tiene 12 tablas.

**Tabla 2.** Entidades del modelo operativo.

| Entidad | Qué representa | Tipo | Campos de GDELT 2.0 de origen |
|---|---|---|---|
| `NOTICIA_EVENTO` | Un evento/noticia y sus métricas de cobertura | Principal | `GLOBALEVENTID`, `SQLDATE`, `DATEADDED`, `IsRootEvent`, `QuadClass`, `GoldsteinScale`, `NumMentions`, `NumSources`, `NumArticles`, `AvgTone` |
| `ACTOR` | Persona, organización o país que participa en eventos | Principal | `Actor{1,2}Code`, `Actor{1,2}Name`, `KnownGroupCode`, `EthnicCode`, `Religion1Code`, `Religion2Code` |
| `UBICACION` | Lugar geográfico (del actor o de la acción) | Principal | `*Geo_FeatureID`, `*Geo_Type`, `*Geo_FullName`, `*Geo_ADM1Code`, `*Geo_ADM2Code`, `*Geo_Lat`, `*Geo_Long` |
| `FUENTE_MEDIO` | Medio que publica artículos | Principal | `MentionSourceName`, `MentionType` (archivo *mentions*) |
| `CODIGO_CAMEO` | Catálogo jerárquico de códigos de acción | Principal | `EventRootCode` → `EventBaseCode` → `EventCode` |
| `PAIS` | País, con código FIPS (geografía) y CAMEO (actores) | Catálogo | `*CountryCode` |
| `TIPO_ACTOR` | Tipos CAMEO de actor (`GOV`, `NGO`, `BUS`, …) y su categoría | Catálogo | `Actor{1,2}Type{1,2,3}Code` |
| `EVENTO_ACTOR` | Participación de un actor en un evento con un rol (1 = Actor1, 2 = Actor2) | Asociativa | Bloques `Actor1*` y `Actor2*` |
| `EVENTO_CATEGORIA` | Clasificación de un evento bajo uno o varios códigos CAMEO | Asociativa | `EventCode` + clasificación editorial |
| `MENCION` | Cada artículo que menciona el evento (seguimiento de eventos en desarrollo) | Asociativa | Archivo *mentions*: `MentionIdentifier`, `MentionTimeDate`, `SentenceID`, `Confidence`, `MentionDocLen`, `MentionDocTone` |
| `ACTOR_TIPO` | Hasta tres tipos por actor, ordenados | Asociativa | `Actor{1,2}Type{1,2,3}Code` |
| `ARTICULO_ORIGEN` | Primer artículo que reportó el evento | Dependiente (1:1) | `SOURCEURL` |

**Mapeo con la consigna.** Las cinco entidades pedidas son las cinco principales: Noticia_Evento → `NOTICIA_EVENTO`, Actor → `ACTOR`, Ubicación → `UBICACION`, Fuente_Medio → `FUENTE_MEDIO` y Código_CAMEO → `CODIGO_CAMEO`.

**Por qué existen las adicionales.**

- `PAIS` y `TIPO_ACTOR` evitan repetir en cada fila códigos y descripciones que casi no cambian.
- `EVENTO_ACTOR`, `EVENTO_CATEGORIA`, `MENCION` y `ACTOR_TIPO` resuelven relaciones M:N, que un modelo relacional no puede representar con una FK simple.
- `ARTICULO_ORIGEN` separa en una tabla 1:1 la URL del primer artículo y su fuente. Así el evento queda con solo sus atributos propios.

### 2.2 Llaves

**Llave natural vs. sustituta.** Una *llave natural* es un atributo con significado en el negocio que ya identifica a la fila. Un ejemplo es `GlobalEventID`, que GDELT asigna a cada evento. Una *llave sustituta* (*surrogate*) es un identificador artificial, sin significado, generado por la propia base (por ejemplo, con una secuencia). En este modelo, `NOTICIA_EVENTO` usa la sustituta `evento_id` como PK y guarda `global_event_id` como llave única alternativa.

**Tabla 3.** Llaves del modelo operativo.

| Tabla | Columna(s) | Tipo | Comentario |
|---|---|---|---|
| `NOTICIA_EVENTO` | `evento_id` | PK, sustituta | Secuencia interna |
| `NOTICIA_EVENTO` | `global_event_id` | UK, natural | Admite NULL (noticias propias del medio) |
| `NOTICIA_EVENTO` | `ubicacion_accion_id` | FK → `UBICACION` | Opcional |
| `ACTOR` | `actor_id` | PK, sustituta | GDELT no identifica a los actores |
| `ACTOR` | `codigo_cameo`, `nombre` | UK, natural compuesta | Un mismo código puede corresponder a varios nombres |
| `ACTOR` | `pais_id` | FK → `PAIS` | Opcional |
| `UBICACION` | `ubicacion_id` | PK, sustituta | |
| `UBICACION` | `feature_id` | UK, natural | Identificador geográfico de GDELT |
| `UBICACION` | `pais_id` | FK → `PAIS` | Obligatoria |
| `FUENTE_MEDIO` | `fuente_id` | PK, sustituta | |
| `FUENTE_MEDIO` | `dominio` | UK, natural | |
| `PAIS` | `pais_id` | PK, sustituta | |
| `PAIS` | `codigo_fips` / `codigo_cameo` | UK, naturales | Dos codificaciones distintas del mismo país |
| `CODIGO_CAMEO` | `cameo_cod` | PK, natural | |
| `CODIGO_CAMEO` | `cameo_padre_cod` | FK recursiva → `CODIGO_CAMEO` | NULL en los códigos raíz |
| `TIPO_ACTOR` | `tipo_actor_cod` | PK, natural | |
| `ACTOR_TIPO` | `actor_id`, `orden` | PK compuesta | `actor_id` también es FK → `ACTOR` |
| `ACTOR_TIPO` | `tipo_actor_cod` | FK → `TIPO_ACTOR` | |
| `EVENTO_ACTOR` | `evento_id`, `rol` | PK compuesta | `evento_id` también es FK → `NOTICIA_EVENTO` |
| `EVENTO_ACTOR` | `actor_id` / `ubicacion_id` | FK → `ACTOR` / FK → `UBICACION` | La ubicación es opcional |
| `EVENTO_CATEGORIA` | `evento_id`, `cameo_cod` | PK compuesta | Ambas columnas también son FK |
| `MENCION` | `mencion_id` | PK, sustituta | |
| `MENCION` | `evento_id` / `fuente_id` | FK → `NOTICIA_EVENTO` / FK → `FUENTE_MEDIO` | |
| `ARTICULO_ORIGEN` | `evento_id` | PK y FK → `NOTICIA_EVENTO` | Compartir la PK implementa la relación 1:1 |
| `ARTICULO_ORIGEN` | `fuente_id` | FK → `FUENTE_MEDIO` | |

**Por qué `evento_id` (sustituta) y no `GlobalEventID` (natural) como PK de `NOTICIA_EVENTO`.**

1. **La asigna un sistema externo.** El medio no controla `GlobalEventID`. Si GDELT reprocesa o renumera eventos, la base operativa no debe verse afectada.
2. **No todas las noticias vienen de GDELT.** La redacción registra noticias propias, que no tienen `GlobalEventID`. La columna tiene que admitir NULL, y una PK no puede ser NULL.
3. **La PK se replica en las tablas hijas.** `evento_id` es FK en `EVENTO_ACTOR`, `EVENTO_CATEGORIA`, `MENCION` y `ARTICULO_ORIGEN`. Una llave interna, compacta e inmutable abarata los joins y evita cambios en cascada.
4. **La natural se sigue usando.** `global_event_id` queda como `UNIQUE`. GDELT vuelve a publicar un evento cada vez que se actualizan sus conteos, y esa restricción permite deduplicar y hacer cargas idempotentes (*upserts*). El estándar SQL admite varios NULL en una columna `UNIQUE`, así que las noticias propias no chocan entre sí.

**Otras sustitutas y naturales.** `ACTOR`, `UBICACION`, `FUENTE_MEDIO` y `PAIS` usan sustituta porque su llave natural no sirve bien como PK:

- en `ACTOR` sería compuesta y larga (código + nombre);
- en `UBICACION`, el `FeatureID` es un texto externo;
- en `FUENTE_MEDIO`, el dominio es un texto largo que puede cambiar;
- en `PAIS`, hay dos codificaciones que conviven (FIPS y CAMEO).

`MENCION` usa sustituta porque no tiene un identificador natural simple. `CODIGO_CAMEO` y `TIPO_ACTOR` usan la **llave natural** porque son códigos estándar, cortos, estables y con significado. Una sustituta no aportaría nada y obligaría a hacer un join solo para leer el código.

**Llaves compuestas.**

- `EVENTO_ACTOR (evento_id, rol)`: identifica la participación de un actor en un evento según su rol. Con `rol ∈ {1, 2}`, un evento tiene como máximo dos actores. Se usa el rol y no `actor_id` porque en GDELT el mismo actor puede aparecer como Actor1 y como Actor2 del mismo evento.
- `EVENTO_CATEGORIA (evento_id, cameo_cod)`: identifica cada clasificación de un evento. Impide asignar dos veces el mismo código al mismo evento.
- `ACTOR_TIPO (actor_id, orden)`: identifica la posición (1, 2 o 3) de cada tipo del actor, que corresponde a `Type1Code`, `Type2Code` y `Type3Code`. Conserva el orden de GDELT y limita la cantidad a tres.

### 2.3 Relaciones

La *participación* indica si cada lado está obligado a tener al menos una fila relacionada: "obligatoria" corresponde a mínimo 1 y "opcional" a mínimo 0. En la Tabla 4, "A" es la entidad padre (la referenciada) y "B" la hija (la que tiene la FK).

**Tabla 4.** Relaciones del modelo operativo.

| Entidad A | Entidad B | Card. | Particip. A | Particip. B | Implementación | Lectura |
|---|---|---|---|---|---|---|
| `NOTICIA_EVENTO` | `ARTICULO_ORIGEN` | **1:1** | Obligatoria | Obligatoria | PK compartida (`evento_id`) | Cada evento tiene exactamente un artículo de origen. |
| `NOTICIA_EVENTO` | `EVENTO_ACTOR` | 1:N | Opcional (0..2) | Obligatoria | FK `evento_id` | Un evento tiene entre 0 y 2 actores. |
| `ACTOR` | `EVENTO_ACTOR` | 1:N | Opcional | Obligatoria | FK `actor_id` | Un actor participa en 0 o más eventos. |
| `UBICACION` | `EVENTO_ACTOR` | 1:N | Opcional | Opcional | FK `ubicacion_id` (NULL) | La participación de un actor puede no estar geolocalizada. |
| `UBICACION` | `NOTICIA_EVENTO` | 1:N | Opcional | Opcional | FK `ubicacion_accion_id` (NULL) | Un evento ocurre en 0 o 1 lugar conocido. |
| `PAIS` | `UBICACION` | 1:N | Opcional | Obligatoria | FK `pais_id` | Toda ubicación pertenece a exactamente un país. |
| `PAIS` | `ACTOR` | 1:N | Opcional | Opcional | FK `pais_id` (NULL) | Un actor tiene 0 o 1 país (por ejemplo, una ONG transnacional no tiene). |
| `NOTICIA_EVENTO` | `EVENTO_CATEGORIA` | 1:N | Obligatoria (1..N) | Obligatoria | FK `evento_id` | Un evento tiene uno o más códigos CAMEO. |
| `CODIGO_CAMEO` | `EVENTO_CATEGORIA` | 1:N | Opcional | Obligatoria | FK `cameo_cod` | Un código clasifica a 0 o más eventos. |
| `CODIGO_CAMEO` | `CODIGO_CAMEO` | 1:N recursiva | Opcional | Opcional | FK `cameo_padre_cod` (NULL) | Un código tiene 0 o 1 padre: raíz → base → evento. |
| `NOTICIA_EVENTO` | `MENCION` | 1:N | Obligatoria (1..N) | Obligatoria | FK `evento_id` | Un evento tiene al menos una mención. |
| `FUENTE_MEDIO` | `MENCION` | 1:N | Opcional | Obligatoria | FK `fuente_id` | Toda mención la publica exactamente un medio. |
| `FUENTE_MEDIO` | `ARTICULO_ORIGEN` | 1:N | Opcional | Obligatoria | FK `fuente_id` | Un medio publicó 0 o más artículos de origen. |
| `ACTOR` | `ACTOR_TIPO` | 1:N | Opcional (0..3) | Obligatoria | FK `actor_id` | Un actor tiene entre 0 y 3 tipos. |
| `TIPO_ACTOR` | `ACTOR_TIPO` | 1:N | Opcional | Obligatoria | FK `tipo_actor_cod` | Un tipo se asigna a 0 o más actores. |

**Relaciones M:N resueltas con tablas asociativas.**

- `NOTICIA_EVENTO` M:N `ACTOR`, mediante `EVENTO_ACTOR` (con atributo de relación `rol`).
- `NOTICIA_EVENTO` M:N `CODIGO_CAMEO`, mediante `EVENTO_CATEGORIA` (con `es_principal`, `origen` y `fecha_asignacion`).
- `NOTICIA_EVENTO` M:N `FUENTE_MEDIO`, mediante `MENCION`: un evento es cubierto por muchos medios y un medio cubre muchos eventos.
- `ACTOR` M:N `TIPO_ACTOR`, mediante `ACTOR_TIPO`.

**Datos vacíos en GDELT.** `Actor2` viene vacío con frecuencia, y a veces también `Actor1`. Por eso la participación del evento en `EVENTO_ACTOR` es opcional (0..2). Las geografías (`Actor*Geo_*`, `ActionGeo_*`) también pueden faltar, así que las FK hacia `UBICACION` admiten NULL.

**Origen de las categorías múltiples.** GDELT trae **un solo `EventCode`** por evento. La relación M:N con `CODIGO_CAMEO` surge de la **clasificación editorial del medio**: al cargar un evento, su `EventCode` se registra con `origen = 'GDELT'` y `es_principal = TRUE`, y los editores agregan códigos adicionales con `origen = 'editor'`.

## 3. Actividad 2: Modelo dimensional (OLAP)

### 3.1 Esquema estrella

Referencia: `TP2.drawio`, página 2.

**Grano.** Cada fila de `FACT_COBERTURA_EVENTO` es una noticia/evento registrado en el OLTP (`evento_id`), venga de GDELT o sea propio del medio, con los conteos de su última actualización.

**Tabla 5.** Dimensiones del esquema estrella.

| Dimensión | Rol (FK en el hecho) | Atributos principales |
|---|---|---|
| `DIM_TIEMPO` | `fecha_publicacion_key` (`DATEADDED`) y `fecha_evento_key` (`SQLDATE`) | `anio`, `trimestre`, `mes`, `anio_mes`, `dia_semana`, `es_fin_de_semana`, `es_festivo`, `tipo_dia` |
| `DIM_ACTOR` | `actor1_key` y `actor2_key` | `nombre`, `pais_nombre`, `tipo_actor_cod`, `tipo_actor_desc`, `categoria_actor`, `tipo2_cod`, `tipo3_cod` |
| `DIM_UBICACION` | `ubicacion_accion_key` | `nombre_completo`, `tipo_geo`, `adm1_cod`, `pais_cod`, `pais_nombre`, `region`, `latitud`, `longitud` |
| `DIM_CAMEO` | `cameo_principal_key` | `evento_cod`/`desc`, `base_cod`/`desc`, `raiz_cod`/`desc`, `quad_class`, `quad_class_desc`, `goldstein_base` |
| `DIM_FUENTE` | `fuente_origen_key` | `dominio`, `nombre_medio`, `tipo_fuente` |
| `DIM_GRUPO_CATEGORIAS` | `grupo_categorias_key` | `cantidad_categorias`, `codigos_cameo`, `es_multitematico` (se conecta con `DIM_CAMEO` a través de `BRIDGE_EVENTO_CAMEO`) |

- **Dimensiones de rol (*role-playing*).** Una misma tabla física se usa con más de un significado. `DIM_TIEMPO` se referencia dos veces: una para la fecha de publicación y otra para la fecha del evento. `DIM_ACTOR` también se referencia dos veces, para Actor1 y Actor2. Así no hace falta duplicar las dimensiones.
- **Dimensiones degeneradas.** `evento_id` (PK del hecho) y `global_event_id` son identificadores sin tabla de dimensión propia. `evento_id` es el grano. `global_event_id` es NULL en las noticias propias y permite rastrear cada fila hasta su evento en GDELT.
- **Miembro -1 "Desconocido".** Cada dimensión tiene una fila con llave -1. Los hechos sin Actor2, sin ubicación, etc. apuntan a ella. Así el hecho no tiene FK nulas y los `JOIN` internos no pierden filas.

**Tabla 6.** Métricas de `FACT_COBERTURA_EVENTO`.

| Métrica | Origen GDELT | Tipo | Cómo se agrega |
|---|---|---|---|
| `cantidad_eventos` | Constante 1 | Aditiva | `SUM` = cantidad de eventos |
| `num_menciones` | `NumMentions` | Aditiva | `SUM` |
| `num_fuentes` | `NumSources` | Aditiva | `SUM`: cuenta pares fuente-evento, no fuentes distintas |
| `num_articulos` | `NumArticles` | Aditiva | `SUM` |
| `tono_x_menciones` | `AvgTone × NumMentions` (se calcula en el ETL) | Aditiva | `SUM`, como numerador del tono ponderado |
| `tono_promedio` | `AvgTone` | No aditiva | `AVG`; conviene usar el tono ponderado |
| `escala_goldstein` | `GoldsteinScale` | No aditiva | `AVG`, `MIN`, `MAX` |

Ninguna métrica es semi-aditiva: no hay saldos ni stocks que se puedan sumar en unas dimensiones pero no en el tiempo.

**Tono ponderado.** Promediar `tono_promedio` le da el mismo peso a un evento con 2 menciones que a uno con 2 000. Para evitarlo, el hecho guarda `tono_x_menciones`, que es aditiva, y el tono de cualquier grupo se calcula como

`tono ponderado = SUM(tono_x_menciones) / SUM(num_menciones)`

Este cálculo equivale a promediar el tono por mención y no por evento.

**Por qué todas las dimensiones usan llave sustituta.**

- Independiza al almacén de las llaves del OLTP y de GDELT.
- Permite versionar atributos (dimensiones lentamente cambiantes, SCD) sin romper los hechos históricos.
- Permite tener el miembro -1.
- Da joins sobre enteros compactos.

`fecha_key` es una sustituta "inteligente" en formato `AAAAMMDD`: sigue siendo un entero, pero se puede leer y sirve para particionar por fecha.

### 3.2 Dimensión temporal

**Tabla 7.** Atributos de `DIM_TIEMPO`.

| Atributo | Tipo | Descripción |
|---|---|---|
| `fecha_key` | INT (PK) | Sustituta inteligente `AAAAMMDD`, por ejemplo 20250324 |
| `fecha` | DATE | Fecha calendario |
| `anio` | SMALLINT | Año |
| `trimestre` | SMALLINT | 1 a 4 |
| `mes` | SMALLINT | 1 a 12 |
| `nombre_mes` | VARCHAR(12) | Enero … Diciembre |
| `anio_mes` | CHAR(7) | `AAAA-MM`, para agrupar por mes sin mezclar años |
| `dia_mes` | SMALLINT | 1 a 31 |
| `dia_semana` | SMALLINT | 1 = lunes … 7 = domingo (ISO 8601) |
| `nombre_dia_semana` | VARCHAR(10) | Lunes … Domingo |
| `es_fin_de_semana` | BOOLEAN | Sábado o domingo |
| `es_festivo` | BOOLEAN | Feriado nacional |
| `nombre_festivo` | VARCHAR(100) | Nombre del feriado; NULL si no lo es |
| `tipo_dia` | VARCHAR(15) | `Festivo`, `Fin de semana` o `Hábil` |

`tipo_dia` resume la pregunta 3 en una sola columna, con esta precedencia: si la fecha es feriado es `Festivo`, aunque caiga en fin de semana; si no, sábado y domingo son `Fin de semana`; el resto es `Hábil`.

**Feriados.** Se toman los de **Argentina**, país sede del medio. En la carga se pueden obtener con la librería `holidays` de Python. La librería incluye los feriados trasladables, los días no laborables y los "feriados con fines turísticos". *Limitación:* GDELT cubre eventos de todo el mundo, y un día hábil en Argentina puede ser feriado en el país donde ocurre el evento. *Alternativa:* modelar los feriados por país, con una tabla de feriados indexada por fecha y país, y cruzarla con el país de la ubicación del evento.

### 3.3 Tabla puente

**Concepto.** Una noticia puede clasificarse bajo varios códigos CAMEO. Entre el hecho y `DIM_CAMEO` hay entonces una relación M:N, que un esquema estrella no puede representar con una FK simple. La solución tiene tres partes:

1. **Grupo de categorías.** Cada combinación distinta de códigos es una fila de `DIM_GRUPO_CATEGORIAS`. El hecho apunta a su grupo con una sola FK (`grupo_categorias_key`). Si varias noticias tienen la misma combinación, comparten el grupo, y la puente no crece con la cantidad de hechos.
2. **Tabla puente.** `BRIDGE_EVENTO_CAMEO (grupo_categorias_key, cameo_key)` lista los códigos de cada grupo, con PK compuesta.
3. **Factor de ponderación.** Si un evento tiene *n* códigos y se agrupa por código, el evento aparece *n* veces y sus métricas se suman *n* veces. Esto es el **doble conteo**. `factor_ponderacion = 1/n` reparte la métrica entre los códigos, de modo que el total vuelve a ser el del hecho. Como el factor se guarda como `DECIMAL(5,4)`, 1/3 queda en 0,3333. Para que los factores de un grupo sumen exactamente 1, el código principal absorbe el redondeo (0,3334).

La columna `es_principal` coincide con `cameo_principal_key` del hecho. Esa FK directa sirve para los análisis que usan un solo código por evento y no necesitan pasar por la puente.

**Implementación.** La consigna pide diseñar **e implementar** la tabla puente. Se usa **DuckDB**, una base analítica embebida que corre en memoria y no requiere servidor. Se implementan `DIM_CAMEO`, `DIM_GRUPO_CATEGORIAS` y `BRIDGE_EVENTO_CAMEO` completas. `FACT_COBERTURA_EVENTO` se implementa en versión **mínima**, solo con las columnas del diagrama que la demostración necesita.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_colwidth", None)
con = duckdb.connect()  # base en memoria

con.execute("""
CREATE SCHEMA dw;
CREATE TABLE dw.DIM_CAMEO (
    cameo_key       INT          PRIMARY KEY,
    evento_cod      VARCHAR(4)   NOT NULL UNIQUE,
    evento_desc     VARCHAR(255) NOT NULL,
    base_cod        VARCHAR(3)   NOT NULL,
    base_desc       VARCHAR(255) NOT NULL,
    raiz_cod        CHAR(2)      NOT NULL,
    raiz_desc       VARCHAR(100) NOT NULL,
    quad_class      SMALLINT     NOT NULL,
    quad_class_desc VARCHAR(40)  NOT NULL,
    goldstein_base  DECIMAL(4,1) NOT NULL
);
CREATE TABLE dw.DIM_GRUPO_CATEGORIAS (
    grupo_categorias_key INT          PRIMARY KEY,
    cantidad_categorias  SMALLINT     NOT NULL,
    codigos_cameo        VARCHAR(100) NOT NULL,
    es_multitematico     BOOLEAN      NOT NULL
);
CREATE TABLE dw.BRIDGE_EVENTO_CAMEO (
    grupo_categorias_key INT          NOT NULL REFERENCES dw.DIM_GRUPO_CATEGORIAS (grupo_categorias_key),
    cameo_key            INT          NOT NULL REFERENCES dw.DIM_CAMEO (cameo_key),
    factor_ponderacion   DECIMAL(5,4) NOT NULL,
    es_principal         BOOLEAN      NOT NULL,
    PRIMARY KEY (grupo_categorias_key, cameo_key)
);
-- Versión mínima del hecho: solo las columnas que usa la demostración
CREATE TABLE dw.FACT_COBERTURA_EVENTO (
    evento_id            BIGINT PRIMARY KEY,   -- dimensión degenerada (grano)
    global_event_id      BIGINT,               -- dimensión degenerada; NULL en noticias propias
    cameo_principal_key  INT    NOT NULL REFERENCES dw.DIM_CAMEO (cameo_key),
    grupo_categorias_key INT    NOT NULL REFERENCES dw.DIM_GRUPO_CATEGORIAS (grupo_categorias_key),
    cantidad_eventos     INT    NOT NULL,
    num_menciones        INT    NOT NULL,
    num_articulos        INT    NOT NULL
);
""")
print("Tablas del almacén:", [t for (t,) in con.execute(
    "SELECT table_name FROM duckdb_tables() WHERE schema_name = 'dw' ORDER BY table_name").fetchall()])

Tablas del almacén: ['BRIDGE_EVENTO_CAMEO', 'DIM_CAMEO', 'DIM_GRUPO_CATEGORIAS', 'FACT_COBERTURA_EVENTO']


**Datos sintéticos.** Los datos siguientes son **ilustrativos**: no provienen de GDELT. Los códigos son códigos CAMEO reales con su descripción traducida del manual CAMEO. Los valores de Goldstein son los de referencia de la escala, y el grupo `quad_class` se deriva de la raíz (01–04: cooperación verbal; 05–08: cooperación material; 09–13: conflicto verbal; 14–20: conflicto material). El evento 8 es una noticia propia del medio, con `global_event_id` NULL.

In [2]:
from decimal import Decimal

# Códigos CAMEO usados: (código, descripción, raíz, descripción de la raíz, Goldstein)
CAMEO = [
    ("010", "Hacer una declaración (no especificada)",             "01", "Hacer una declaración pública",  0.0),
    ("036", "Expresar intención de reunirse o negociar",           "03", "Expresar intención de cooperar", 4.0),
    ("042", "Realizar una visita",                                 "04", "Consultar",                      1.9),
    ("043", "Recibir una visita",                                  "04", "Consultar",                      2.8),
    ("057", "Firmar un acuerdo formal",                            "05", "Cooperar diplomáticamente",      8.0),
    ("112", "Acusar (no especificado)",                            "11", "Desaprobar",                    -2.0),
    ("141", "Manifestarse o movilizarse",                          "14", "Protestar",                     -6.5),
    ("173", "Arrestar, detener o acusar judicialmente",            "17", "Coercionar",                    -5.0),
    ("190", "Usar fuerza militar convencional (no especificado)",  "19", "Combatir",                     -10.0),
]
QUAD = {1: "Cooperación verbal", 2: "Cooperación material", 3: "Conflicto verbal", 4: "Conflicto material"}


def quad_class(raiz):
    r = int(raiz)
    return 1 if r <= 4 else 2 if r <= 8 else 3 if r <= 13 else 4


dim_cameo = pd.DataFrame([
    {"cameo_key": i, "evento_cod": cod, "evento_desc": desc,
     "base_cod": cod, "base_desc": desc,          # con 3 dígitos, el código base es el propio código
     "raiz_cod": raiz, "raiz_desc": raiz_desc,
     "quad_class": quad_class(raiz), "quad_class_desc": QUAD[quad_class(raiz)],
     "goldstein_base": gs}
    for i, (cod, desc, raiz, raiz_desc, gs) in enumerate(CAMEO, start=1)
])
cameo_key = dict(zip(dim_cameo["evento_cod"], dim_cameo["cameo_key"]))

# Eventos: (evento_id, global_event_id, códigos con el principal primero, menciones, artículos)
EVENTOS = [
    (1, 1200000101, ["042", "043"],        40, 12),  # visita de Estado: visita + recepción
    (2, 1200000102, ["057"],               25,  8),
    (3, 1200000103, ["141", "173", "112"], 120, 35), # protesta con detenciones y acusaciones
    (4, 1200000104, ["036"],               15,  5),
    (5, 1200000105, ["190"],               60, 20),
    (6, 1200000106, ["141", "173"],        30,  9),
    (7, 1200000107, ["042", "043"],        18,  6),  # comparte grupo con el evento 1
    (8, None,       ["010", "112"],        10,  3),  # noticia propia del medio
]

# Un grupo por combinación distinta de códigos (principal primero, el resto ordenado)
grupos, filas_puente, filas_hecho = {}, [], []
for evento_id, gid, cods, menciones, articulos in EVENTOS:
    clave = (cods[0],) + tuple(sorted(cods[1:]))
    if clave not in grupos:
        grupos[clave] = len(grupos) + 1
        n = len(clave)
        base = (Decimal(1) / n).quantize(Decimal("0.0001"))
        for j, cod in enumerate(clave):
            factor = Decimal(1) - base * (n - 1) if j == 0 else base  # el principal absorbe el redondeo
            filas_puente.append({"grupo_categorias_key": grupos[clave], "cameo_key": cameo_key[cod],
                                 "factor_ponderacion": str(factor), "es_principal": j == 0})
    filas_hecho.append({"evento_id": evento_id, "global_event_id": gid,
                        "cameo_principal_key": cameo_key[cods[0]], "grupo_categorias_key": grupos[clave],
                        "cantidad_eventos": 1, "num_menciones": menciones, "num_articulos": articulos})

dim_grupo = pd.DataFrame([
    {"grupo_categorias_key": k, "cantidad_categorias": len(c), "codigos_cameo": "|".join(c),
     "es_multitematico": len(c) > 1}
    for c, k in grupos.items()
])
puente = pd.DataFrame(filas_puente)
hecho = pd.DataFrame(filas_hecho).astype({"global_event_id": "Int64"})

for tabla, df in [("DIM_CAMEO", dim_cameo), ("DIM_GRUPO_CATEGORIAS", dim_grupo), ("FACT_COBERTURA_EVENTO", hecho)]:
    con.register("tmp_df", df)
    con.execute(f"INSERT INTO dw.{tabla} BY NAME SELECT * FROM tmp_df")
con.register("tmp_df", puente)
con.execute("""INSERT INTO dw.BRIDGE_EVENTO_CAMEO
               SELECT grupo_categorias_key, cameo_key, CAST(factor_ponderacion AS DECIMAL(5,4)), es_principal
               FROM tmp_df""")

# Tabla 8: contenido de la puente, con el grupo y el código legibles
con.sql("""
    SELECT g.grupo_categorias_key AS grupo, g.codigos_cameo, g.cantidad_categorias AS n,
           c.evento_cod, c.evento_desc, b.factor_ponderacion, b.es_principal
    FROM dw.BRIDGE_EVENTO_CAMEO b
    JOIN dw.DIM_GRUPO_CATEGORIAS g USING (grupo_categorias_key)
    JOIN dw.DIM_CAMEO c USING (cameo_key)
    ORDER BY grupo, b.es_principal DESC, c.evento_cod
""").df()

,grupo,codigos_cameo,n,evento_cod,evento_desc,factor_ponderacion,es_principal
0,1,042|043,2,042,Realizar una visita,0.5000,True
1,1,042|043,2,043,Recibir una visita,0.5000,False
2,2,057,1,057,Firmar un acuerdo formal,1.0000,True
3,3,141|112|173,3,141,Manifestarse o movilizarse,0.3334,True
4,3,141|112|173,3,112,Acusar (no especificado),0.3333,False
5,3,141|112|173,3,173,"Arrestar, detener o acusar judicialmente",0.3333,False
6,4,036,1,036,Expresar intención de reunirse o negociar,1.0000,True
7,5,190,1,190,Usar fuerza militar convencional (no especificado),1.0000,True
8,6,141|173,2,141,Manifestarse o movilizarse,0.5000,True
9,6,141|173,2,173,"Arrestar, detener o acusar judicialmente",0.5000,False


**Tabla 8.** Contenido de `BRIDGE_EVENTO_CAMEO` (salida anterior). Hay 8 eventos y 7 grupos, porque los eventos 1 y 7 comparten la combinación `042|043`.

**Consultas.** La consulta (a) suma las menciones por código **sin ponderar**, y la consulta (b) las suma **ponderadas** por `factor_ponderacion`.

In [3]:
con.sql("""
    SELECT c.evento_cod AS codigo, c.evento_desc AS descripcion,
           CAST(SUM(f.num_menciones) AS BIGINT)        AS a_menciones_sin_ponderar,
           SUM(f.num_menciones * b.factor_ponderacion) AS b_menciones_ponderadas
    FROM dw.FACT_COBERTURA_EVENTO f
    JOIN dw.BRIDGE_EVENTO_CAMEO b USING (grupo_categorias_key)
    JOIN dw.DIM_CAMEO c USING (cameo_key)
    GROUP BY c.evento_cod, c.evento_desc
    ORDER BY a_menciones_sin_ponderar DESC
""").df()

,codigo,descripcion,a_menciones_sin_ponderar,b_menciones_ponderadas
0,173,"Arrestar, detener o acusar judicialmente",150,54.996
1,141,Manifestarse o movilizarse,150,55.008
2,112,Acusar (no especificado),130,44.996
3,190,Usar fuerza militar convencional (no especificado),60,60.000
4,042,Realizar una visita,58,29.000
5,043,Recibir una visita,58,29.000
6,057,Firmar un acuerdo formal,25,25.000
7,036,Expresar intención de reunirse o negociar,15,15.000
8,010,Hacer una declaración (no especificada),10,5.000


**Tabla 9.** Menciones por código CAMEO, sin ponderar (a) y ponderadas (b) (salida anterior).

**Verificación.** La suma ponderada (b) tiene que coincidir con el total del hecho. La suma sin ponderar (a) tiene que superarlo, porque cuenta cada evento multi-temático una vez por cada código.

In [4]:
totales = con.sql("""
    SELECT (SELECT SUM(num_menciones) FROM dw.FACT_COBERTURA_EVENTO) AS total_hecho,
           SUM(f.num_menciones)                                    AS total_a_sin_ponderar,
           SUM(f.num_menciones * b.factor_ponderacion)             AS total_b_ponderado
    FROM dw.FACT_COBERTURA_EVENTO f
    JOIN dw.BRIDGE_EVENTO_CAMEO b USING (grupo_categorias_key)
""").fetchone()
total_hecho, total_a, total_b = totales

assert total_b == total_hecho, "La suma ponderada debe coincidir con el total del hecho"
assert total_a > total_hecho, "La suma sin ponderar debe superar al total del hecho"
print(f"Total del hecho:        {total_hecho}")
print(f"(a) sin ponderar:       {total_a}  -> doble conteo de {total_a - total_hecho} menciones")
print(f"(b) ponderado:          {total_b}  -> coincide con el hecho")

Total del hecho:        318
(a) sin ponderar:       656  -> doble conteo de 338 menciones
(b) ponderado:          318.0000  -> coincide con el hecho


(c) **Eventos multi-temáticos** y sus códigos. Esta consulta responde la pregunta de negocio 4.

In [5]:
con.sql("""
    SELECT f.evento_id, f.global_event_id, g.cantidad_categorias,
           string_agg(c.evento_cod || ' ' || c.evento_desc, '; '
                      ORDER BY b.es_principal DESC, c.evento_cod) AS codigos_principal_primero,
           f.num_menciones
    FROM dw.FACT_COBERTURA_EVENTO f
    JOIN dw.DIM_GRUPO_CATEGORIAS g USING (grupo_categorias_key)
    JOIN dw.BRIDGE_EVENTO_CAMEO b USING (grupo_categorias_key)
    JOIN dw.DIM_CAMEO c USING (cameo_key)
    WHERE g.es_multitematico
    GROUP BY f.evento_id, f.global_event_id, g.cantidad_categorias, f.num_menciones
    ORDER BY g.cantidad_categorias DESC, f.evento_id
""").df()

,evento_id,global_event_id,cantidad_categorias,codigos_principal_primero,num_menciones
0,3,1200000103,3,"141 Manifestarse o movilizarse; 112 Acusar (no especificado); 173 Arrestar, detener o acusar judicialmente",120
1,1,1200000101,2,042 Realizar una visita; 043 Recibir una visita,40
2,6,1200000106,2,"141 Manifestarse o movilizarse; 173 Arrestar, detener o acusar judicialmente",30
3,7,1200000107,2,042 Realizar una visita; 043 Recibir una visita,18
4,8,<NA>,2,010 Hacer una declaración (no especificada); 112 Acusar (no especificado),10


**Tabla 10.** Eventos multi-temáticos (salida anterior).

## 4. Trazabilidad con las preguntas de negocio

**Tabla 11.** Cómo responde el modelo dimensional a cada pregunta.

| Pregunta | Tablas / atributos | Métrica | Agregación |
|---|---|---|---|
| 1. Impacto por país y mes | `DIM_UBICACION.pais_nombre`, `DIM_TIEMPO.anio_mes` (fecha de publicación), `DIM_CAMEO.raiz_desc` (tema) o `evento_id` (noticia) | `num_menciones`, `num_articulos` | `SUM` y ranking por país y mes |
| 2. Tono según el tipo de actor | `DIM_ACTOR.categoria_actor`, con los roles Actor1 y Actor2 | `tono_x_menciones`, `num_menciones` | Tono ponderado = `SUM(tono_x_menciones) / SUM(num_menciones)` |
| 3. Intensidad por tipo de día | `DIM_TIEMPO.tipo_dia` (fecha de publicación) | `cantidad_eventos`, `num_articulos` | `SUM` dividido por la cantidad de días de cada tipo |
| 4. Eventos multi-temáticos | `DIM_GRUPO_CATEGORIAS.es_multitematico` → `BRIDGE_EVENTO_CAMEO` → `DIM_CAMEO` | `num_menciones` (ponderada por `factor_ponderacion`) | Filtro y listado de códigos |

Las consultas 1 a 3 se presentan sin ejecutar, porque el informe no carga datos reales de GDELT. La consulta 4 ya se ejecutó en la sección 3.3.

**Pregunta 1.** Temas con más menciones por país y mes. Para rankear noticias en lugar de temas, se reemplaza `c.raiz_desc` por `f.evento_id`.

```sql
WITH impacto AS (
    SELECT u.pais_nombre, t.anio_mes, c.raiz_desc AS tema,
           SUM(f.num_menciones) AS menciones,
           SUM(f.num_articulos) AS articulos
    FROM dw.FACT_COBERTURA_EVENTO f
    JOIN dw.DIM_UBICACION u ON u.ubicacion_key = f.ubicacion_accion_key
    JOIN dw.DIM_TIEMPO    t ON t.fecha_key     = f.fecha_publicacion_key
    JOIN dw.DIM_CAMEO     c ON c.cameo_key     = f.cameo_principal_key
    WHERE u.ubicacion_key <> -1                      -- excluye ubicación desconocida
    GROUP BY u.pais_nombre, t.anio_mes, c.raiz_desc
)
SELECT *,
       RANK() OVER (PARTITION BY pais_nombre, anio_mes ORDER BY menciones DESC) AS ranking
FROM impacto
QUALIFY ranking <= 10
ORDER BY pais_nombre, anio_mes, ranking;
```

**Pregunta 2.** Tono ponderado por categoría de actor. Cada evento aporta una fila por cada actor presente.

```sql
WITH participaciones AS (
    SELECT actor1_key AS actor_key, num_menciones, tono_x_menciones FROM dw.FACT_COBERTURA_EVENTO
    UNION ALL
    SELECT actor2_key AS actor_key, num_menciones, tono_x_menciones FROM dw.FACT_COBERTURA_EVENTO
)
SELECT a.categoria_actor,
       COUNT(*)                                            AS participaciones,
       SUM(p.tono_x_menciones) / NULLIF(SUM(p.num_menciones), 0) AS tono_ponderado
FROM participaciones p
JOIN dw.DIM_ACTOR a USING (actor_key)
WHERE a.actor_key <> -1                              -- excluye actor ausente
GROUP BY a.categoria_actor
ORDER BY tono_ponderado;
```

**Pregunta 3.** Intensidad por tipo de día, normalizada por la cantidad de días de cada tipo. En 2025, en Argentina, hubo 244 días hábiles y 20 feriados: comparar totales sin normalizar solo mostraría esa diferencia. Los días se cuentan en `DIM_TIEMPO` y no en el hecho, para que también cuenten los días sin eventos.

```sql
WITH dias AS (
    SELECT tipo_dia, COUNT(*) AS cantidad_dias
    FROM dw.DIM_TIEMPO
    WHERE fecha BETWEEN DATE '2025-01-01' AND DATE '2025-12-31'
    GROUP BY tipo_dia
),
flujo AS (
    SELECT t.tipo_dia,
           SUM(f.cantidad_eventos) AS eventos,
           SUM(f.num_articulos)    AS articulos
    FROM dw.FACT_COBERTURA_EVENTO f
    JOIN dw.DIM_TIEMPO t ON t.fecha_key = f.fecha_publicacion_key
    WHERE t.fecha BETWEEN DATE '2025-01-01' AND DATE '2025-12-31'
    GROUP BY t.tipo_dia
)
SELECT d.tipo_dia, d.cantidad_dias,
       COALESCE(fl.eventos, 0)   / d.cantidad_dias AS eventos_por_dia,
       COALESCE(fl.articulos, 0) / d.cantidad_dias AS articulos_por_dia
FROM dias d
LEFT JOIN flujo fl USING (tipo_dia)
ORDER BY eventos_por_dia DESC;
```

**Pregunta 4.** Ver la consulta (c) de la sección 3.3.

## 5. Supuestos y limitaciones

- **Grano en la última actualización.** El hecho guarda solo los conteos más recientes de cada evento. No se puede ver cómo creció la cobertura de un evento con el tiempo; para eso haría falta un hecho de menciones o *snapshots* periódicos.
- **Feriados de un solo país.** Solo se consideran los de Argentina (ver 3.2).
- **Cardinalidades mínimas.** Que cada evento tenga al menos una mención, al menos una categoría y exactamente un artículo de origen no se puede declarar con FK. Lo garantiza la aplicación con transacciones.
- **Dimensiones lentamente cambiantes.** Se asume SCD tipo 1 (sobrescribir). Para conservar la historia de un actor (por ejemplo, un cambio de país o de tipo) habría que usar SCD tipo 2.
- **Datos sintéticos.** Los datos de la sección 3.3 son ilustrativos. Los valores de Goldstein son los de referencia y no provienen de una descarga de GDELT.

## 6. Referencias

- GDELT Project. *GDELT 2.0 Event Database Codebook (V2.0)*. https://data.gdeltproject.org/documentation/GDELT-Event_Codebook-V2.0.pdf
- Schrodt, P. A. *CAMEO: Conflict and Mediation Event Observations — Event and Actor Codebook (v1.1b3)*. https://gdeltproject.org/data/documentation/CAMEO.Manual.1.1b3.pdf
- GDELT Project. *Lista maestra de archivos de GDELT 2.0* (actualizada cada 15 minutos). https://data.gdeltproject.org/gdeltv2/masterfilelist.txt